# Binomial Option Pricing Model\n\n**Level:** Intermediate\n**Concepts Covered:** 6\n\n---\n\n## Overview\n\nThis comprehensive notebook covers binomial option pricing model with detailed explanations, mathematical derivations, Python implementations, and practical examples.

## 1. Introduction

### Why the Binomial Model Matters

The **binomial option pricing model**, developed by Cox, Ross, and Rubinstein (1979), is one of the most important tools in quantitative finance. Unlike the Black-Scholes model, which assumes continuous trading, the binomial model uses discrete time steps, making it:

- **Intuitive**: Based on simple up/down price movements
- **Flexible**: Can price American options (early exercise)
- **Educational**: Reveals the fundamental principles of risk-neutral valuation
- **Practical**: Computationally efficient and numerically stable

### Real-World Applications

- **American Option Pricing**: Banks and hedge funds use binomial trees for options with early exercise features
- **Convertible Bonds**: Valuing securities with embedded optionality
- **Employee Stock Options**: Accounting for vesting schedules and early exercise
- **Exotic Options**: Path-dependent and barrier options
- **Risk Management**: Computing option Greeks through tree perturbations

### Learning Objectives

By the end of this notebook, you will be able to:

1. **Construct** one-period and multi-period binomial trees for stock price evolution
2. **Derive** risk-neutral probabilities and explain their financial interpretation
3. **Implement** binomial pricing algorithms for European and American options
4. **Compute** option Greeks using the binomial framework
5. **Demonstrate** convergence of binomial prices to Black-Scholes as steps increase
6. **Compare** binomial and Black-Scholes approaches for different option types

### Prerequisites

- Understanding of basic options (calls/puts, payoffs)
- Familiarity with probability theory and expectation
- Python programming (NumPy, Matplotlib)
- Optional: Knowledge of Black-Scholes formula

### Historical Context

The binomial model revolutionized derivatives pricing when introduced in 1979. It provided:
- An **alternative** to the Black-Scholes PDE approach
- A **computational method** accessible before advanced numerical techniques
- **Insight** into the replication argument underpinning derivatives pricing
- **Foundation** for modern lattice and tree-based methods

**Estimated Time**: 90-120 minutes

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

print('[OK] Libraries imported successfully')

## 2. One-Period Binomial Model

### Theory

The one-period binomial model is the building block for multi-period trees. Consider a stock with current price $S_0$ that can move to one of two states after time $\Delta t$:

**Price Evolution**:
- **Up state**: $S_u = S_0 \cdot u$ with probability $p$
- **Down state**: $S_d = S_0 \cdot d$ with probability $(1-p)$

where $u > 1$ (up factor) and $d < 1$ (down factor).

### No-Arbitrage Condition

For the model to be arbitrage-free, we must have:
$$
d < e^{r\Delta t} < u
$$

If this condition is violated, riskless arbitrage opportunities exist:
- If $e^{r\Delta t} \geq u$: Borrow money, buy stock, guaranteed profit
- If $e^{r\Delta t} \leq d$: Short stock, invest at risk-free rate, guaranteed profit

### Risk-Neutral Probability

The **risk-neutral probability** $q$ is not the true probability of the up move, but rather the probability that makes the expected return equal to the risk-free rate:

$$
q = \frac{e^{r\Delta t} - d}{u - d}
$$

**Key Insight**: Under risk-neutral measure, the discounted expected stock price equals the current price:
$$
S_0 = e^{-r\Delta t}[q \cdot S_u + (1-q) \cdot S_d]
$$

### Option Valuation

For a European call option with strike $K$:

**Step 1**: Calculate payoffs at expiration:
$$
\begin{align*}
C_u &= \max(S_u - K, 0) \\
C_d &= \max(S_d - K, 0)
\end{align*}
$$

**Step 2**: Discount expected payoff under risk-neutral measure:
$$
C_0 = e^{-r\Delta t}[q \cdot C_u + (1-q) \cdot C_d]
$$

This formula gives the **no-arbitrage price** of the option.

### Replication Argument

The option can be replicated using a portfolio of $\Delta$ shares and $B$ bonds:

$$
\Delta = \frac{C_u - C_d}{S_u - S_d}, \quad B = e^{-r\Delta t} \frac{u \cdot C_d - d \cdot C_u}{u - d}
$$

The replication cost equals the option price: $C_0 = \Delta \cdot S_0 + B$

This connects binomial pricing to **delta hedging** in practice.

### Implementation: One-Period Model

Let's implement the one-period binomial model with a concrete numerical example.

In [ ]:
def one_period_binomial(S0: float, K: float, r: float, T: float, u: float, d: float, 
                        option_type: str = 'call') -> dict:
    """
    Price a European option using one-period binomial model.
    
    Parameters
    ----------
    S0 : float
        Initial stock price
    K : float
        Strike price
    r : float
        Risk-free rate (annualized)
    T : float
        Time to expiration (years)
    u : float
        Up factor (multiplicative)
    d : float
        Down factor (multiplicative)
    option_type : str, default 'call'
        Type of option ('call' or 'put')
    
    Returns
    -------
    dict
        Dictionary containing option price, risk-neutral probability, 
        payoffs, and replicating portfolio
    """
    # Calculate stock prices in up and down states
    Su = S0 * u
    Sd = S0 * d
    
    # Calculate option payoffs at expiration
    if option_type.lower() == 'call':
        Cu = max(Su - K, 0)
        Cd = max(Sd - K, 0)
    elif option_type.lower() == 'put':
        Cu = max(K - Su, 0)
        Cd = max(K - Sd, 0)
    else:
        raise ValueError("option_type must be 'call' or 'put'")
    
    # Risk-neutral probability
    q = (np.exp(r * T) - d) / (u - d)
    
    # Check no-arbitrage condition
    if not (d < np.exp(r * T) < u):
        print(f"WARNING: No-arbitrage condition violated! d={d:.4f}, e^rT={np.exp(r*T):.4f}, u={u:.4f}")
    
    # Option value (discounted expected payoff under risk-neutral measure)
    C0 = np.exp(-r * T) * (q * Cu + (1 - q) * Cd)
    
    # Replicating portfolio
    delta = (Cu - Cd) / (Su - Sd)  # Number of shares
    B = np.exp(-r * T) * (u * Cd - d * Cu) / (u - d)  # Bond position
    
    # Verify replication
    replication_cost = delta * S0 + B
    
    return {
        'option_price': C0,
        'risk_neutral_prob': q,
        'payoff_up': Cu,
        'payoff_down': Cd,
        'stock_up': Su,
        'stock_down': Sd,
        'delta': delta,
        'bond': B,
        'replication_cost': replication_cost
    }


# Example: Price a call option
print("=" * 60)
print("ONE-PERIOD BINOMIAL MODEL: EUROPEAN CALL OPTION")
print("=" * 60)

# Parameters
S0 = 100    # Current stock price
K = 100     # Strike price (ATM)
r = 0.05    # Risk-free rate (5%)
T = 1.0     # Time to expiration (1 year)
u = 1.2     # Up factor (20% increase)
d = 0.9     # Down factor (10% decrease)

result = one_period_binomial(S0, K, r, T, u, d, option_type='call')

print(f"\nInput Parameters:")
print(f"  S0 = ${S0:.2f}")
print(f"  K = ${K:.2f}")
print(f"  r = {r:.1%}")
print(f"  T = {T:.2f} years")
print(f"  u = {u:.2f} (stock up to ${result['stock_up']:.2f})")
print(f"  d = {d:.2f} (stock down to ${result['stock_down']:.2f})")

print(f"\nRisk-Neutral Probability:")
print(f"  q = {result['risk_neutral_prob']:.4f} (probability of up move)")
print(f"  1-q = {1-result['risk_neutral_prob']:.4f} (probability of down move)")

print(f"\nOption Payoffs at Expiration:")
print(f"  If stock goes UP: Call payoff = max(${result['stock_up']:.2f} - ${K:.2f}, 0) = ${result['payoff_up']:.2f}")
print(f"  If stock goes DOWN: Call payoff = max(${result['stock_down']:.2f} - ${K:.2f}, 0) = ${result['payoff_down']:.2f}")

print(f"\nOption Price Today:")
print(f"  C0 = e^(-{r}*{T}) * [{result['risk_neutral_prob']:.4f} * ${result['payoff_up']:.2f} + {1-result['risk_neutral_prob']:.4f} * ${result['payoff_down']:.2f}]")
print(f"  C0 = ${result['option_price']:.4f}")

print(f"\nReplicating Portfolio:")
print(f"  Delta (shares to hold): {result['delta']:.4f}")
print(f"  Bond position: ${result['bond']:.4f}")
print(f"  Replication cost: ${result['replication_cost']:.4f}")
print(f"  Matches option price: {np.isclose(result['option_price'], result['replication_cost'])}")

print("\n" + "=" * 60)
print('[OK] One-Period Binomial Model implementation complete')

In [ ]:
# Visualization: One-Period Binomial Tree

def plot_one_period_tree(S0, u, d, K, option_type='call'):
    """Plot one-period binomial tree showing stock and option values."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
    
    # Stock price tree
    Su = S0 * u
    Sd = S0 * d
    
    # Option payoffs
    if option_type == 'call':
        Cu = max(Su - K, 0)
        Cd = max(Sd - K, 0)
    else:
        Cu = max(K - Su, 0)
        Cd = max(K - Sd, 0)
    
    # Plot 1: Stock Price Tree
    ax1.plot([0, 1], [S0, Su], 'b-', linewidth=2, marker='o', markersize=12, label='Up move')
    ax1.plot([0, 1], [S0, Sd], 'r-', linewidth=2, marker='o', markersize=12, label='Down move')
    
    # Annotations for stock prices
    ax1.text(0, S0, f'  S₀ = ${S0:.2f}', fontsize=12, va='center', fontweight='bold')
    ax1.text(1, Su, f'  Sᵤ = ${Su:.2f}', fontsize=11, va='center', color='blue')
    ax1.text(1, Sd, f'  Sᵤ = ${Sd:.2f}', fontsize=11, va='center', color='red')
    
    ax1.set_xlim(-0.2, 1.3)
    ax1.set_ylim(Sd * 0.9, Su * 1.1)
    ax1.set_xlabel('Time Step', fontsize=12)
    ax1.set_ylabel('Stock Price ($)', fontsize=12)
    ax1.set_title('Stock Price Evolution', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Plot 2: Option Value Tree
    result = one_period_binomial(S0, K, 0.05, 1.0, u, d, option_type)
    C0 = result['option_price']
    
    ax2.plot([0, 1], [C0, Cu], 'b-', linewidth=2, marker='s', markersize=12, label='Up move')
    ax2.plot([0, 1], [C0, Cd], 'r-', linewidth=2, marker='s', markersize=12, label='Down move')
    
    # Annotations for option values
    ax2.text(0, C0, f'  C₀ = ${C0:.4f}', fontsize=12, va='center', fontweight='bold')
    ax2.text(1, Cu, f'  Cᵤ = ${Cu:.2f}', fontsize=11, va='center', color='blue')
    ax2.text(1, Cd, f'  Cᵤ = ${Cd:.2f}', fontsize=11, va='center', color='red')
    
    ax2.set_xlim(-0.2, 1.3)
    ax2.set_ylim(min(C0, Cd) * 0.8 - 1, max(C0, Cu) * 1.2 + 1)
    ax2.set_xlabel('Time Step', fontsize=12)
    ax2.set_ylabel(f'{option_type.capitalize()} Option Value ($)', fontsize=12)
    ax2.set_title(f'{option_type.capitalize()} Option Valuation', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

# Generate visualization
plot_one_period_tree(S0=100, u=1.2, d=0.9, K=100, option_type='call')
print('[OK] One-period binomial tree visualization complete')

### Practical Example\n\nLet's apply one-period binomial model to a real-world scenario...

## 3. Multi-Period Binomial Trees\n\n### Theory\n\nDetailed explanation of multi-period binomial trees...

In [ ]:
def build_stock_tree(S0: float, u: float, d: float, n: int) -> np.ndarray:
    """
    Build binomial tree for stock prices.
    
    Parameters
    ----------
    S0 : float
        Initial stock price
    u : float
        Up factor
    d : float
        Down factor
    n : int
        Number of time steps
    
    Returns
    -------
    np.ndarray
        2D array where tree[i, j] is stock price at step i, node j
    """
    tree = np.zeros((n + 1, n + 1))
    
    for i in range(n + 1):
        for j in range(i + 1):
            # j up-moves, (i-j) down-moves
            tree[i, j] = S0 * (u ** j) * (d ** (i - j))
    
    return tree


def binomial_tree_option(S0: float, K: float, T: float, r: float, sigma: float, 
                         n: int, option_type: str = 'call', 
                         american: bool = False) -> dict:
    """
    Price European or American option using binomial tree.
    
    Parameters
    ----------
    S0 : float
        Initial stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility (annualized)
    n : int
        Number of time steps
    option_type : str
        'call' or 'put'
    american : bool
        If True, price American option with early exercise
    
    Returns
    -------
    dict
        Contains option price, stock tree, option tree
    """
    # Time step
    dt = T / n
    
    # CRR parameterization
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    q = (np.exp(r * dt) - d) / (u - d)
    discount = np.exp(-r * dt)
    
    # Build stock price tree
    stock_tree = build_stock_tree(S0, u, d, n)
    
    # Initialize option value tree
    option_tree = np.zeros((n + 1, n + 1))
    
    # Terminal payoffs
    for j in range(n + 1):
        if option_type.lower() == 'call':
            option_tree[n, j] = max(stock_tree[n, j] - K, 0)
        elif option_type.lower() == 'put':
            option_tree[n, j] = max(K - stock_tree[n, j], 0)
        else:
            raise ValueError("option_type must be 'call' or 'put'")
    
    # Backward induction
    for i in range(n - 1, -1, -1):
        for j in range(i + 1):
            # Continuation value (hold option)
            continuation = discount * (q * option_tree[i + 1, j + 1] + 
                                      (1 - q) * option_tree[i + 1, j])
            
            if american:
                # Early exercise value
                if option_type.lower() == 'call':
                    exercise = max(stock_tree[i, j] - K, 0)
                else:
                    exercise = max(K - stock_tree[i, j], 0)
                
                # Take maximum (American option)
                option_tree[i, j] = max(continuation, exercise)
            else:
                # European option (no early exercise)
                option_tree[i, j] = continuation
    
    return {
        'option_price': option_tree[0, 0],
        'stock_tree': stock_tree,
        'option_tree': option_tree,
        'u': u,
        'd': d,
        'q': q,
        'dt': dt
    }


# Example: Price European call with multi-period tree
print("=" * 70)
print("MULTI-PERIOD BINOMIAL TREE: EUROPEAN CALL")
print("=" * 70)

S0 = 100
K = 100
T = 1.0
r = 0.05
sigma = 0.20
n = 5  # 5 time steps

result = binomial_tree_option(S0, K, T, r, sigma, n, option_type='call', american=False)

print(f"\nParameters:")
print(f"  S0 = ${S0:.2f}, K = ${K:.2f}, T = {T} year, r = {r:.1%}, σ = {sigma:.1%}")
print(f"  Number of steps: {n}")
print(f"  Time step: Δt = {result['dt']:.4f}")
print(f"  CRR parameters: u = {result['u']:.4f}, d = {result['d']:.4f}, q = {result['q']:.4f}")

print(f"\nStock Price Tree (first 3 steps):")
for i in range(min(4, n + 1)):
    print(f"  Step {i}: ", end="")
    for j in range(i + 1):
        print(f"${result['stock_tree'][i, j]:7.2f} ", end="")
    print()

print(f"\nOption Value Tree (first 3 steps):")
for i in range(min(4, n + 1)):
    print(f"  Step {i}: ", end="")
    for j in range(i + 1):
        print(f"${result['option_tree'][i, j]:7.4f} ", end="")
    print()

print(f"\n\nEuropean Call Price: ${result['option_price']:.4f}")
print("=" * 70)

print('[OK] Multi-Period Binomial Trees implementation complete')

In [ ]:
def visualize_binomial_tree(stock_tree: np.ndarray, option_tree: np.ndarray, 
                           n_display: int = None) -> None:
    """
    Visualize multi-period binomial trees for stock and option values.
    
    Parameters
    ----------
    stock_tree : np.ndarray
        Stock price tree from binomial_tree_option
    option_tree : np.ndarray
        Option value tree from binomial_tree_option
    n_display : int, optional
        Number of steps to display (default: all steps up to 8)
    """
    n = stock_tree.shape[0] - 1
    if n_display is None:
        n_display = min(n, 8)  # Limit display to 8 steps for readability
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12))
    
    # Plot 1: Stock Price Tree
    for i in range(n_display):
        for j in range(i + 1):
            # Current node
            x_curr, y_curr = i, j - i/2
            
            # Draw node
            ax1.plot(x_curr, y_curr, 'o', markersize=20, color='steelblue', 
                    markeredgecolor='navy', markeredgewidth=2, zorder=3)
            
            # Add stock price label
            ax1.text(x_curr, y_curr, f'${stock_tree[i, j]:.1f}', 
                    ha='center', va='center', fontsize=9, fontweight='bold', 
                    color='white', zorder=4)
            
            # Draw edges to next nodes
            if i < n_display - 1:
                # Up edge
                x_next_up, y_next_up = i + 1, (j + 1) - (i + 1)/2
                ax1.plot([x_curr, x_next_up], [y_curr, y_next_up], 
                        'b-', linewidth=2, alpha=0.4, zorder=1)
                
                # Down edge
                x_next_down, y_next_down = i + 1, j - (i + 1)/2
                ax1.plot([x_curr, x_next_down], [y_curr, y_next_down], 
                        'r-', linewidth=2, alpha=0.4, zorder=1)
    
    # Add final nodes if not displayed
    if n_display <= n:
        i = n_display
        for j in range(i + 1):
            x_curr, y_curr = i, j - i/2
            ax1.plot(x_curr, y_curr, 'o', markersize=20, color='steelblue',
                    markeredgecolor='navy', markeredgewidth=2, zorder=3)
            ax1.text(x_curr, y_curr, f'${stock_tree[i, j]:.1f}',
                    ha='center', va='center', fontsize=9, fontweight='bold',
                    color='white', zorder=4)
    
    ax1.set_xlabel('Time Step', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Node Position', fontsize=12, fontweight='bold')
    ax1.set_title('Stock Price Binomial Tree', fontsize=14, fontweight='bold', pad=20)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(-0.5, n_display + 0.5)
    
    # Plot 2: Option Value Tree
    for i in range(n_display):
        for j in range(i + 1):
            # Current node
            x_curr, y_curr = i, j - i/2
            
            # Draw node
            ax2.plot(x_curr, y_curr, 's', markersize=20, color='darkgreen',
                    markeredgecolor='darkslategray', markeredgewidth=2, zorder=3)
            
            # Add option value label
            ax2.text(x_curr, y_curr, f'${option_tree[i, j]:.2f}',
                    ha='center', va='center', fontsize=9, fontweight='bold',
                    color='white', zorder=4)
            
            # Draw edges to next nodes
            if i < n_display - 1:
                # Up edge
                x_next_up, y_next_up = i + 1, (j + 1) - (i + 1)/2
                ax2.plot([x_curr, x_next_up], [y_curr, y_next_up],
                        'b-', linewidth=2, alpha=0.4, zorder=1)
                
                # Down edge
                x_next_down, y_next_down = i + 1, j - (i + 1)/2
                ax2.plot([x_curr, x_next_down], [y_curr, y_next_down],
                        'r-', linewidth=2, alpha=0.4, zorder=1)
    
    # Add final nodes if not displayed
    if n_display <= n:
        i = n_display
        for j in range(i + 1):
            x_curr, y_curr = i, j - i/2
            ax2.plot(x_curr, y_curr, 's', markersize=20, color='darkgreen',
                    markeredgecolor='darkslategray', markeredgewidth=2, zorder=3)
            ax2.text(x_curr, y_curr, f'${option_tree[i, j]:.2f}',
                    ha='center', va='center', fontsize=9, fontweight='bold',
                    color='white', zorder=4)
    
    ax2.set_xlabel('Time Step', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Node Position', fontsize=12, fontweight='bold')
    ax2.set_title('Option Value Binomial Tree', fontsize=14, fontweight='bold', pad=20)
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(-0.5, n_display + 0.5)
    
    plt.tight_layout()
    plt.show()


# Visualize the binomial tree from previous example
print("Visualizing 5-step binomial tree:")
visualize_binomial_tree(result['stock_tree'], result['option_tree'], n_display=5)
print('[OK] Multi-period binomial tree visualization complete')

### Practical Example: SPY Option Pricing

Let's apply the multi-period binomial model to price a real SPY (S&P 500 ETF) call option.

In [ ]:
# Practical Example: Pricing SPY Call Option
print("=" * 70)
print("PRACTICAL EXAMPLE: SPY CALL OPTION PRICING")
print("=" * 70)

# Realistic SPY parameters (as of typical market conditions)
S0_spy = 450.00      # SPY current price
K_spy = 455.00       # Strike price (slightly OTM)
T_spy = 0.25         # 3 months to expiration
r_spy = 0.045        # Risk-free rate (4.5%)
sigma_spy = 0.18     # Historical volatility (18% annualized)

# Price with different number of steps to show accuracy improvement
steps_list = [10, 25, 50, 100]

print(f"\nOption Parameters:")
print(f"  Underlying: SPY at ${S0_spy:.2f}")
print(f"  Strike: ${K_spy:.2f} (call option)")
print(f"  Time to Expiration: {T_spy*252:.0f} trading days ({T_spy*12:.1f} months)")
print(f"  Risk-Free Rate: {r_spy:.2%}")
print(f"  Volatility: {sigma_spy:.1%}")

print(f"\nBinomial Pricing with Different Step Counts:")
print(f"{'Steps':<10} {'Option Price':<15} {'Computation Time':<20}")
print("-" * 45)

import time

for n_steps in steps_list:
    start_time = time.time()
    result_spy = binomial_tree_option(S0_spy, K_spy, T_spy, r_spy, sigma_spy, 
                                     n_steps, option_type='call', american=False)
    elapsed = time.time() - start_time
    
    print(f"{n_steps:<10} ${result_spy['option_price']:<14.4f} {elapsed*1000:<20.2f} ms")

# Use 50 steps for detailed analysis
n_optimal = 50
result_spy_detailed = binomial_tree_option(S0_spy, K_spy, T_spy, r_spy, sigma_spy,
                                          n_optimal, option_type='call', american=False)

print(f"\n\nDetailed Results (n={n_optimal} steps):")
print(f"  European Call Price: ${result_spy_detailed['option_price']:.4f}")
print(f"  Tree Parameters: u={result_spy_detailed['u']:.6f}, d={result_spy_detailed['d']:.6f}")
print(f"  Risk-Neutral Probability: q={result_spy_detailed['q']:.6f}")
print(f"  Time Step: Δt={result_spy_detailed['dt']:.6f} years")

print("\n" + "=" * 70)
print('[OK] SPY option pricing example complete')

## 4. Risk-Neutral Valuation\n\n### Theory\n\nDetailed explanation of risk-neutral valuation...

### Mathematical Formulation\n\nThe mathematical framework for risk-neutral valuation is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Visualization for Risk-Neutral Valuation

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Risk-Neutral Valuation')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Practical example for Risk-Neutral Valuation

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

### Implementation: American vs European Options

Let's compare American and European put options to see the value of early exercise.

In [ ]:
def visualize_early_exercise_boundary(S0: float, K: float, T: float, r: float,
                                     sigma: float, n: int) -> None:
    """
    Visualize the early exercise boundary for American put option.
    
    Parameters
    ----------
    S0 : float
        Initial stock price
    K : float
        Strike price
    T : float
        Time to expiration
    r : float
        Risk-free rate
    sigma : float
        Volatility
    n : int
        Number of time steps
    """
    # Price American put
    american_result = binomial_tree_option(S0, K, T, r, sigma, n, 
                                          option_type='put', american=True)
    
    stock_tree = american_result['stock_tree']
    option_tree = american_result['option_tree']
    dt = T / n
    
    # Find early exercise boundary
    boundary_prices = []
    boundary_times = []
    
    for i in range(n):
        # For each time step, find the highest stock price where early exercise is optimal
        boundary_found = False
        for j in range(i, -1, -1):  # Start from highest stock price at this time
            stock_price = stock_tree[i, j]
            intrinsic = max(K - stock_price, 0)
            option_value = option_tree[i, j]
            
            # Check if early exercise is optimal
            if np.isclose(option_value, intrinsic, rtol=1e-5) and intrinsic > 0.01:
                if not boundary_found:
                    boundary_prices.append(stock_price)
                    boundary_times.append(i * dt)
                    boundary_found = True
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Early Exercise Boundary
    if len(boundary_times) > 0:
        ax1.plot(boundary_times, boundary_prices, 'ro-', linewidth=2, 
                markersize=6, label='Early Exercise Boundary')
        ax1.axhline(y=K, color='green', linestyle='--', linewidth=2, 
                   label=f'Strike Price (K=${K:.0f})')
        ax1.fill_between(boundary_times, boundary_prices, K, 
                        alpha=0.3, color='red', label='Early Exercise Region')
    
    ax1.set_xlabel('Time to Expiration (years)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Stock Price ($)', fontsize=12, fontweight='bold')
    ax1.set_title('American Put: Early Exercise Boundary', 
                 fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='best', fontsize=10)
    ax1.set_xlim([0, T])
    
    # Add annotation
    if len(boundary_prices) > 0:
        mid_idx = len(boundary_prices) // 2
        ax1.annotate('Exercise put if\nstock price below\nthis boundary',
                    xy=(boundary_times[mid_idx], boundary_prices[mid_idx]),
                    xytext=(boundary_times[mid_idx] + 0.1, boundary_prices[mid_idx] - 5),
                    arrowprops=dict(arrowstyle='->', color='red', lw=2),
                    fontsize=10, color='darkred', fontweight='bold')
    
    # Plot 2: American vs European Put Values
    european_result = binomial_tree_option(S0, K, T, r, sigma, n,
                                          option_type='put', american=False)
    
    # Get terminal stock prices and values
    terminal_stocks = stock_tree[n, :]
    american_terminal = option_tree[n, :]
    european_terminal = european_result['option_tree'][n, :]
    
    # Sort by stock price for plotting
    sort_idx = np.argsort(terminal_stocks)
    terminal_stocks_sorted = terminal_stocks[sort_idx]
    american_terminal_sorted = american_terminal[sort_idx]
    european_terminal_sorted = european_terminal[sort_idx]
    
    ax2.plot(terminal_stocks_sorted, american_terminal_sorted, 'ro-', 
            linewidth=2, markersize=5, label='American Put')
    ax2.plot(terminal_stocks_sorted, european_terminal_sorted, 'bs-',
            linewidth=2, markersize=5, label='European Put')
    
    # Plot intrinsic value
    intrinsic_values = np.maximum(K - terminal_stocks_sorted, 0)
    ax2.plot(terminal_stocks_sorted, intrinsic_values, 'g--',
            linewidth=2, label='Intrinsic Value')
    
    ax2.set_xlabel('Stock Price at Expiration ($)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Option Value ($)', fontsize=12, fontweight='bold')
    ax2.set_title('American vs European Put Values', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='best', fontsize=10)
    
    plt.tight_layout()
    plt.show()


# Visualize early exercise boundary
print("\nVisualizing Early Exercise Boundary for American Put:")
visualize_early_exercise_boundary(S0=100, K=100, T=1.0, r=0.05, sigma=0.25, n=50)

print('[OK] Early Exercise Boundary visualization complete')

## 6. Convergence to Black-Scholes\n\n### Theory\n\nDetailed explanation of convergence to black-scholes...

In [ ]:
# Visualization for Convergence to Black-Scholes

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Convergence to Black-Scholes')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

def black_scholes_price(S0: float, K: float, T: float, r: float, sigma: float,
                       option_type: str = 'call') -> float:
    """
    Calculate Black-Scholes option price.
    
    Parameters
    ----------
    S0 : float
        Initial stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility
    option_type : str
        'call' or 'put'
    
    Returns
    -------
    float
        Black-Scholes option price
    """
    if T <= 0:
        # At expiration
        if option_type.lower() == 'call':
            return max(S0 - K, 0)
        else:
            return max(K - S0, 0)
    
    # Calculate d1 and d2
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    # Calculate option price
    if option_type.lower() == 'call':
        price = S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type.lower() == 'put':
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)
    else:
        raise ValueError("option_type must be 'call' or 'put'")
    
    return price


def analyze_convergence(S0: float, K: float, T: float, r: float, sigma: float,
                       option_type: str = 'call', max_steps: int = 500) -> dict:
    """
    Analyze convergence of binomial model to Black-Scholes.
    
    Parameters
    ----------
    S0 : float
        Initial stock price
    K : float
        Strike price
    T : float
        Time to expiration
    r : float
        Risk-free rate
    sigma : float
        Volatility
    option_type : str
        'call' or 'put'
    max_steps : int
        Maximum number of steps to test
    
    Returns
    -------
    dict
        Contains step counts, binomial prices, BS price, and errors
    """
    # Calculate Black-Scholes price
    bs_price = black_scholes_price(S0, K, T, r, sigma, option_type)
    
    # Test different step counts
    step_counts = [5, 10, 20, 30, 50, 75, 100, 150, 200, 300, max_steps]
    binomial_prices = []
    errors = []
    abs_errors = []
    
    for n in step_counts:
        result = binomial_tree_option(S0, K, T, r, sigma, n, option_type, american=False)
        binom_price = result['option_price']
        
        binomial_prices.append(binom_price)
        error = binom_price - bs_price
        errors.append(error)
        abs_errors.append(abs(error))
    
    return {
        'step_counts': step_counts,
        'binomial_prices': binomial_prices,
        'bs_price': bs_price,
        'errors': errors,
        'abs_errors': abs_errors
    }


# Demonstrate convergence
print("=" * 80)
print("CONVERGENCE TO BLACK-SCHOLES")
print("=" * 80)

S0 = 100
K = 100
T = 1.0
r = 0.05
sigma = 0.20

conv_results = analyze_convergence(S0, K, T, r, sigma, option_type='call', max_steps=500)

print(f"\nParameters:")
print(f"  S0 = ${S0:.2f}, K = ${K:.2f}, T = {T} year")
print(f"  r = {r:.1%}, σ = {sigma:.1%}")

print(f"\n\nBlack-Scholes Price: ${conv_results['bs_price']:.6f}")

print(f"\n\nConvergence Analysis:")
print(f"{'Steps':<10} {'Binomial Price':<18} {'Error':<15} {'|Error|':<15}")
print("-" * 60)

for i, n in enumerate(conv_results['step_counts']):
    print(f"{n:<10} ${conv_results['binomial_prices'][i]:<17.6f} "
          f"{conv_results['errors'][i]:>+14.6f} "
          f"{conv_results['abs_errors'][i]:<14.6f}")

print(f"\n\nKey Observations:")
print(f"  • Error at n=10: ${conv_results['abs_errors'][1]:.6f}")
print(f"  • Error at n=100: ${conv_results['abs_errors'][6]:.6f}")
print(f"  • Error at n=500: ${conv_results['abs_errors'][-1]:.6f}")
print(f"  • Improvement (10→100 steps): {conv_results['abs_errors'][1]/conv_results['abs_errors'][6]:.2f}x")
print(f"  • Improvement (100→500 steps): {conv_results['abs_errors'][6]/conv_results['abs_errors'][-1]:.2f}x")

print("\n" + "=" * 80)
print('[OK] Convergence to Black-Scholes implementation complete')

In [ ]:
def visualize_convergence(conv_results: dict) -> None:
    """
    Visualize convergence of binomial model to Black-Scholes.
    
    Parameters
    ----------
    conv_results : dict
        Results from analyze_convergence function
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    step_counts = np.array(conv_results['step_counts'])
    binomial_prices = np.array(conv_results['binomial_prices'])
    bs_price = conv_results['bs_price']
    errors = np.array(conv_results['errors'])
    abs_errors = np.array(conv_results['abs_errors'])
    
    # Plot 1: Binomial vs Black-Scholes Price
    ax1.plot(step_counts, binomial_prices, 'bo-', linewidth=2, markersize=6,
            label='Binomial Model')
    ax1.axhline(y=bs_price, color='red', linestyle='--', linewidth=2,
               label=f'Black-Scholes = ${bs_price:.4f}')
    ax1.fill_between(step_counts, binomial_prices, bs_price, alpha=0.2)
    ax1.set_xlabel('Number of Steps (n)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Option Price ($)', fontsize=12, fontweight='bold')
    ax1.set_title('Binomial Price Convergence to Black-Scholes', 
                 fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='best', fontsize=10)
    ax1.set_xscale('log')
    
    # Plot 2: Pricing Error vs Steps
    ax2.plot(step_counts, errors, 'ro-', linewidth=2, markersize=6)
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax2.set_xlabel('Number of Steps (n)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Pricing Error ($)', fontsize=12, fontweight='bold')
    ax2.set_title('Pricing Error (Binomial - Black-Scholes)', 
                 fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.set_xscale('log')
    
    # Plot 3: Absolute Error vs Steps (log-log scale)
    ax3.loglog(step_counts, abs_errors, 'go-', linewidth=2, markersize=6,
              label='Absolute Error')
    
    # Add theoretical convergence rate O(1/sqrt(n))
    theoretical_rate = abs_errors[0] * (step_counts[0] ** 0.5) / (step_counts ** 0.5)
    ax3.loglog(step_counts, theoretical_rate, 'k--', linewidth=2, alpha=0.5,
              label=r'$O(1/\sqrt{n})$ rate')
    
    ax3.set_xlabel('Number of Steps (n)', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Absolute Error ($)', fontsize=12, fontweight='bold')
    ax3.set_title('Convergence Rate Analysis (log-log scale)', 
                 fontsize=13, fontweight='bold')
    ax3.grid(True, alpha=0.3, which='both')
    ax3.legend(loc='best', fontsize=10)
    
    # Plot 4: Relative Error Percentage
    relative_errors = (abs_errors / bs_price) * 100
    ax4.plot(step_counts, relative_errors, 'mo-', linewidth=2, markersize=6)
    ax4.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, 
               label='1% error threshold')
    ax4.axhline(y=0.1, color='green', linestyle='--', linewidth=1.5,
               label='0.1% error threshold')
    ax4.set_xlabel('Number of Steps (n)', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Relative Error (%)', fontsize=12, fontweight='bold')
    ax4.set_title('Relative Pricing Error', fontsize=13, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    ax4.legend(loc='best', fontsize=10)
    ax4.set_xscale('log')
    ax4.set_yscale('log')
    
    plt.tight_layout()
    plt.show()


# Visualize convergence
print("\nVisualizing Convergence to Black-Scholes:")
visualize_convergence(conv_results)

print('[OK] Convergence visualization complete')

### Mathematical Formulation\n\nThe mathematical framework for greeks from binomial model is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Visualization for Greeks from Binomial Model

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Greeks from Binomial Model')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Option Greeks from Binomial Trees

### Theory: Sensitivity Analysis

**Option Greeks** measure how option prices change with respect to various parameters. The binomial model provides an intuitive way to compute Greeks using **finite differences**.

### Key Greeks

**Delta (Δ)**: Rate of change of option price with respect to stock price
$$
\Delta \approx \frac{\partial C}{\partial S} = \frac{C_u - C_d}{S_u - S_d}
$$

- **Interpretation**: Number of shares to hold in delta-hedging portfolio
- **Range**: 0 to 1 for calls, -1 to 0 for puts
- **At-the-money**: Delta ≈ 0.5 for calls

**Gamma (Γ)**: Rate of change of Delta with respect to stock price
$$
\Gamma \approx \frac{\partial^2 C}{\partial S^2} = \frac{\Delta_u - \Delta_d}{0.5(S_{uu} - S_{dd})}
$$

- **Interpretation**: Convexity of option price, stability of hedge
- **Maximum**: At-the-money options
- **Important for**: Risk management, understanding hedge rebalancing needs

**Theta (Θ)**: Rate of change of option price with respect to time
$$
\Theta \approx \frac{\partial C}{\partial t} = \frac{C_{t+\Delta t} - C_t}{\Delta t}
$$

- **Interpretation**: Time decay (usually negative)
- **Maximum**: At-the-money options near expiration

### Computing Greeks from Binomial Trees

The binomial tree naturally provides finite difference approximations:

1. **Delta**: Use nodes at time step 1
2. **Gamma**: Use nodes at time step 2
3. **Theta**: Compare option values across time steps

These approximations become more accurate as $n$ increases.

### Implementation: Greeks Calculation Functions

In [ ]:
def calculate_greeks_binomial(S0: float, K: float, T: float, r: float, sigma: float,
                             n: int, option_type: str = 'call') -> dict:
    """
    Calculate option Greeks using binomial tree finite differences.
    
    Parameters
    ----------
    S0 : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to expiration (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility
    n : int
        Number of time steps
    option_type : str
        'call' or 'put'
    
    Returns
    -------
    dict
        Dictionary containing delta, gamma, and theta
    """
    # Build tree with current parameters
    result = binomial_tree_option(S0, K, T, r, sigma, n, option_type, american=False)
    option_tree = result['option_tree']
    stock_tree = result['stock_tree']
    dt = result['dt']
    
    # Delta: Finite difference at time step 1
    if n >= 1:
        delta = (option_tree[1, 1] - option_tree[1, 0]) / (stock_tree[1, 1] - stock_tree[1, 0])
    else:
        delta = np.nan
    
    # Gamma: Second order finite difference at time step 2
    if n >= 2:
        delta_up = (option_tree[2, 2] - option_tree[2, 1]) / (stock_tree[2, 2] - stock_tree[2, 1])
        delta_down = (option_tree[2, 1] - option_tree[2, 0]) / (stock_tree[2, 1] - stock_tree[2, 0])
        gamma = (delta_up - delta_down) / (0.5 * (stock_tree[2, 2] - stock_tree[2, 0]))
    else:
        gamma = np.nan
    
    # Theta: Time decay (build a tree with one fewer step)
    if n >= 2:
        result_short = binomial_tree_option(S0, K, T, r, sigma, n-1, option_type, american=False)
        theta = (result_short['option_price'] - option_tree[0, 0]) / dt
    else:
        theta = np.nan
    
    return {
        'delta': delta,
        'gamma': gamma,
        'theta': theta
    }


def compare_american_european(S0: float, K: float, T: float, r: float, sigma: float,
                              n: int, option_type: str = 'call') -> dict:
    """
    Compare American and European option prices.
    
    Parameters
    ----------
    S0 : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to expiration
    r : float
        Risk-free rate
    sigma : float
        Volatility
    n : int
        Number of time steps
    option_type : str
        'call' or 'put'
    
    Returns
    -------
    dict
        Dictionary with European price, American price, and early exercise premium
    """
    european_result = binomial_tree_option(S0, K, T, r, sigma, n, 
                                          option_type, american=False)
    american_result = binomial_tree_option(S0, K, T, r, sigma, n,
                                          option_type, american=True)
    
    european_price = european_result['option_price']
    american_price = american_result['option_price']
    early_exercise_premium = american_price - european_price
    
    return {
        'european_price': european_price,
        'american_price': american_price,
        'early_exercise_premium': early_exercise_premium
    }


def calculate_greeks_across_strikes(S0: float, T: float, r: float, sigma: float, n: int,
                                   option_type: str = 'call', strike_range: tuple = (0.7, 1.3),
                                   n_strikes: int = 31) -> dict:
    """
    Calculate option Greeks across a range of strike prices.
    
    Parameters
    ----------
    S0 : float
        Current stock price
    T : float
        Time to expiration
    r : float
        Risk-free rate
    sigma : float
        Volatility
    n : int
        Number of time steps in binomial tree
    option_type : str
        'call' or 'put'
    strike_range : tuple
        (min_ratio, max_ratio) as fractions of S0
    n_strikes : int
        Number of strikes to evaluate
    
    Returns
    -------
    dict
        Dictionary containing strikes, prices, deltas, gammas, and thetas arrays
    """
    # Generate strike prices
    strikes = np.linspace(strike_range[0] * S0, strike_range[1] * S0, n_strikes)
    
    prices = []
    deltas = []
    gammas = []
    thetas = []
    
    for K in strikes:
        # Calculate option price
        result = binomial_tree_option(S0, K, T, r, sigma, n, option_type, american=False)
        prices.append(result['option_price'])
        
        # Calculate Greeks
        greeks = calculate_greeks_binomial(S0, K, T, r, sigma, n, option_type)
        deltas.append(greeks['delta'])
        gammas.append(greeks['gamma'])
        thetas.append(greeks['theta'])
    
    return {
        'strikes': strikes,
        'prices': np.array(prices),
        'deltas': np.array(deltas),
        'gammas': np.array(gammas),
        'thetas': np.array(thetas)
    }


print('[OK] Greeks calculation helper functions defined')

In [ ]:
# CASE STUDY: AAPL Call Option Pricing
print("=" * 90)
print("COMPREHENSIVE CASE STUDY: APPLE (AAPL) CALL OPTION")
print("=" * 90)

# Market parameters
S0_aapl = 175.00
K_aapl = 180.00
T_aapl = 60 / 365  # 60 days
r_aapl = 0.045
sigma_aapl = 0.22

print(f"\nMarket Data:")
print(f"  Stock: AAPL")
print(f"  Current Price: ${S0_aapl:.2f}")
print(f"  Strike Price: ${K_aapl:.2f} ({'OTM' if S0_aapl < K_aapl else 'ITM'} call)")
print(f"  Days to Expiration: 60 ({T_aapl:.4f} years)")
print(f"  Risk-Free Rate: {r_aapl:.2%}")
print(f"  Implied Volatility: {sigma_aapl:.1%}")

# Step 1: Price with multiple methods
print(f"\n\n{'='*90}")
print("STEP 1: PRICING COMPARISON")
print("=" * 90)

# Binomial with different steps
n_steps_list = [20, 50, 100, 200]
print(f"\nBinomial Model Prices:")
binom_prices = {}
for n in n_steps_list:
    result = binomial_tree_option(S0_aapl, K_aapl, T_aapl, r_aapl, sigma_aapl, n,
                                 option_type='call', american=False)
    binom_prices[n] = result['option_price']
    print(f"  n={n:>3} steps: ${result['option_price']:.4f}")

# Black-Scholes price
bs_price_aapl = black_scholes_price(S0_aapl, K_aapl, T_aapl, r_aapl, sigma_aapl, 'call')
print(f"\nBlack-Scholes Price: ${bs_price_aapl:.4f}")

print(f"\nPricing Differences from Black-Scholes:")
for n in n_steps_list:
    diff = binom_prices[n] - bs_price_aapl
    print(f"  n={n:>3}: {diff:>+.4f} ({abs(diff)/bs_price_aapl*100:.2f}%)")

# Step 2: Greeks Analysis
print(f"\n\n{'='*90}")
print("STEP 2: GREEKS FOR RISK MANAGEMENT")
print("=" * 90)

greeks_aapl = calculate_greeks_binomial(S0_aapl, K_aapl, T_aapl, r_aapl, sigma_aapl, 
                                       100, option_type='call')

print(f"\nOption Greeks (using 100-step binomial tree):")
print(f"  Delta (Δ):  {greeks_aapl['delta']:.4f}")
print(f"    → To hedge 100 contracts, need to short {int(greeks_aapl['delta'] * 100 * 100)} shares")
print(f"  Gamma (Γ):  {greeks_aapl['gamma']:.6f}")
print(f"    → Delta changes by {greeks_aapl['gamma']:.6f} per $1 move in AAPL")
print(f"  Theta (Θ):  {greeks_aapl['theta']:.4f} per year")
print(f"    → Time decay: ${greeks_aapl['theta']/252:.4f} per day")
print(f"    → For 100 contracts: ${greeks_aapl['theta']/252 * 100 * 100:.2f} per day")

# Step 3: American vs European
print(f"\n\n{'='*90}")
print("STEP 3: AMERICAN vs EUROPEAN OPTION")
print("=" * 90)

comparison_aapl = compare_american_european(S0_aapl, K_aapl, T_aapl, r_aapl, 
                                           sigma_aapl, 100, option_type='call')

print(f"\nPricing Results:")
print(f"  European Call: ${comparison_aapl['european_price']:.4f}")
print(f"  American Call: ${comparison_aapl['american_price']:.4f}")
print(f"  Early Exercise Premium: ${comparison_aapl['early_exercise_premium']:.6f}")

if np.isclose(comparison_aapl['early_exercise_premium'], 0, atol=1e-4):
    print(f"\n  ✓ Confirmed: For non-dividend stock, American call = European call")
    print(f"    Early exercise is never optimal (as predicted by theory)")

# Step 4: Volatility Sensitivity
print(f"\n\n{'='*90}")
print("STEP 4: VOLATILITY SENSITIVITY ANALYSIS")
print("=" * 90)

volatilities = np.array([0.15, 0.18, 0.20, 0.22, 0.25, 0.28, 0.30])
vega_analysis = []

print(f"\nOption Price vs Implied Volatility:")
print(f"{'Volatility':<15} {'Binomial Price':<18} {'BS Price':<15} {'Difference':<12}")
print("-" * 65)

for vol in volatilities:
    binom_result = binomial_tree_option(S0_aapl, K_aapl, T_aapl, r_aapl, vol, 100,
                                       option_type='call', american=False)
    bs_result = black_scholes_price(S0_aapl, K_aapl, T_aapl, r_aapl, vol, 'call')
    vega_analysis.append((vol, binom_result['option_price'], bs_result))
    diff = binom_result['option_price'] - bs_result
    print(f"{vol:>6.1%}         ${binom_result['option_price']:<16.4f} ${bs_result:<14.4f} {diff:>+11.4f}")

# Step 5: Practical Trading Implications
print(f"\n\n{'='*90}")
print("STEP 5: PRACTICAL TRADING IMPLICATIONS")
print("=" * 90)

print(f"\nKey Insights for AAPL ${K_aapl:.0f} Call (60 days to expiration):")
print(f"\n1. FAIR VALUE")
print(f"   • Theoretical price: ${bs_price_aapl:.2f} per contract")
print(f"   • Total investment for 10 contracts: ${bs_price_aapl * 10 * 100:,.2f}")

print(f"\n2. DELTA HEDGING")
print(f"   • Current delta: {greeks_aapl['delta']:.4f}")
print(f"   • Position: Long 10 call contracts (1,000 shares exposure)")
print(f"   • Delta-neutral hedge: Short {int(greeks_aapl['delta'] * 1000)} shares of AAPL")
print(f"   • Hedge value: ${greeks_aapl['delta'] * 1000 * S0_aapl:,.2f}")

print(f"\n3. RISK METRICS")
print(f"   • Gamma: {greeks_aapl['gamma']:.6f}")
print(f"   • If AAPL moves $1, delta changes by {greeks_aapl['gamma']:.6f}")
print(f"   • For $5 move, need to rehedge ~{int(greeks_aapl['gamma'] * 5 * 1000)} shares")

print(f"\n4. TIME DECAY")
print(f"   • Theta: ${greeks_aapl['theta']/252:.4f} per day")
print(f"   • 10 contracts lose ${abs(greeks_aapl['theta']/252 * 10 * 100):.2f}/day to time decay")
print(f"   • Over one week: ${abs(greeks_aapl['theta']/252 * 10 * 100 * 5):.2f} (assuming 5 trading days)")

print(f"\n5. BREAKEVEN ANALYSIS")
breakeven = K_aapl + bs_price_aapl
profit_5pct = (S0_aapl * 1.05 - K_aapl - bs_price_aapl) * 10 * 100
print(f"   • Breakeven price at expiration: ${breakeven:.2f}")
print(f"   • Current price: ${S0_aapl:.2f}")
print(f"   • Need {(breakeven - S0_aapl)/S0_aapl * 100:.2f}% move to breakeven")
print(f"   • If AAPL rises 5% to ${S0_aapl * 1.05:.2f}: Profit = ${profit_5pct:,.2f}")

print(f"\n\n{'='*90}")
print("CASE STUDY COMPLETE")
print("=" * 90)
print('[OK] AAPL option pricing case study complete')

def visualize_greeks(S0: float, T: float, r: float, sigma: float, n: int) -> None:
    """
    Visualize option Greeks across different strike prices.
    
    Parameters
    ----------
    S0 : float
        Current stock price
    T : float
        Time to expiration
    r : float
        Risk-free rate
    sigma : float
        Volatility
    n : int
        Number of time steps in binomial tree
    """
    # Calculate Greeks for calls and puts
    call_greeks = calculate_greeks_across_strikes(S0, T, r, sigma, n, 
                                                  option_type='call',
                                                  strike_range=(0.7, 1.3),
                                                  n_strikes=31)
    
    put_greeks = calculate_greeks_across_strikes(S0, T, r, sigma, n,
                                                option_type='put',
                                                strike_range=(0.7, 1.3),
                                                n_strikes=31)
    
    strikes = call_greeks['strikes']
    moneyness = strikes / S0
    
    # Create visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # Plot 1: Option Prices
    ax1.plot(strikes, call_greeks['prices'], 'b-', linewidth=2, label='Call Price')
    ax1.plot(strikes, put_greeks['prices'], 'r-', linewidth=2, label='Put Price')
    ax1.axvline(x=S0, color='green', linestyle='--', linewidth=1.5, 
               label=f'Current Price S=${S0:.0f}')
    ax1.set_xlabel('Strike Price ($)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Option Price ($)', fontsize=12, fontweight='bold')
    ax1.set_title('Option Prices vs Strike', fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='best', fontsize=10)
    
    # Plot 2: Delta
    ax2.plot(strikes, call_greeks['deltas'], 'b-', linewidth=2, label='Call Delta')
    ax2.plot(strikes, put_greeks['deltas'], 'r-', linewidth=2, label='Put Delta')
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax2.axvline(x=S0, color='green', linestyle='--', linewidth=1.5,
               label=f'S=${S0:.0f}')
    ax2.set_xlabel('Strike Price ($)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Delta', fontsize=12, fontweight='bold')
    ax2.set_title('Delta vs Strike', fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='best', fontsize=10)
    ax2.set_ylim([-1.1, 1.1])
    
    # Add annotations
    ax2.annotate('Deep ITM\ncall: Δ→1', xy=(strikes[0], call_greeks['deltas'][0]),
                xytext=(strikes[0] + 5, 0.8), fontsize=9,
                arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))
    ax2.annotate('Deep ITM\nput: Δ→-1', xy=(strikes[-1], put_greeks['deltas'][-1]),
                xytext=(strikes[-1] - 5, -0.8), fontsize=9,
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
    
    # Plot 3: Gamma
    ax3.plot(strikes, call_greeks['gammas'], 'b-', linewidth=2, label='Call Gamma')
    ax3.plot(strikes, put_greeks['gammas'], 'r--', linewidth=2, label='Put Gamma')
    ax3.axvline(x=S0, color='green', linestyle='--', linewidth=1.5,
               label=f'S=${S0:.0f}')
    ax3.set_xlabel('Strike Price ($)', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Gamma', fontsize=12, fontweight='bold')
    ax3.set_title('Gamma vs Strike', fontsize=13, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.legend(loc='best', fontsize=10)
    
    # Find and annotate max gamma
    max_gamma_idx = np.argmax(call_greeks['gammas'])
    ax3.annotate(f'Max Γ\n(ATM)', 
                xy=(strikes[max_gamma_idx], call_greeks['gammas'][max_gamma_idx]),
                xytext=(strikes[max_gamma_idx] + 10, call_greeks['gammas'][max_gamma_idx] * 0.8),
                fontsize=10, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='darkblue', lw=2))
    
    # Plot 4: Theta
    ax4.plot(strikes, call_greeks['thetas'], 'b-', linewidth=2, label='Call Theta')
    ax4.plot(strikes, put_greeks['thetas'], 'r-', linewidth=2, label='Put Theta')
    ax4.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax4.axvline(x=S0, color='green', linestyle='--', linewidth=1.5,
               label=f'S=${S0:.0f}')
    ax4.set_xlabel('Strike Price ($)', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Theta (per year)', fontsize=12, fontweight='bold')
    ax4.set_title('Theta vs Strike', fontsize=13, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    ax4.legend(loc='best', fontsize=10)
    
    # Add annotation
    ax4.text(0.05, 0.95, 'Negative theta:\ntime decay', 
            transform=ax4.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', 
            facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()


# Visualize Greeks
print("\nVisualizing Option Greeks Across Strikes:")
visualize_greeks(S0=100, T=0.5, r=0.05, sigma=0.25, n=100)

print('[OK] Greeks visualization complete')

## Summary and Key Takeaways

### What We Learned

This notebook provided a comprehensive treatment of the binomial option pricing model, from foundational concepts to advanced applications. Here are the key takeaways:

#### 1. Core Concepts
- **Binomial trees** provide a discrete-time framework for modeling stock price evolution
- **Risk-neutral valuation** eliminates the need to estimate expected returns
- **Backward induction** computes option values by working from expiration to present
- **Replication argument** shows options can be synthesized using stocks and bonds

#### 2. Practical Implementation
- **Cox-Ross-Rubinstein parameterization** ensures convergence to Black-Scholes
- **Computational efficiency**: O(n²) complexity for n-step trees
- **Flexibility**: Handles American options, dividends, and exotic features
- **Accuracy**: Converges to Black-Scholes at rate O(1/√n)

#### 3. American vs European Options
- American calls on non-dividend stocks = European calls (no early exercise)
- American puts can have significant early exercise premium
- Early exercise boundary depends on strike, volatility, rates, and time
- Binomial model naturally handles early exercise through backward induction

#### 4. Greeks and Risk Management
- **Delta**: Hedge ratio, typically 0-1 for calls, -1-0 for puts
- **Gamma**: Convexity measure, maximum at-the-money
- **Theta**: Time decay, typically negative for long options
- Finite differences from binomial tree provide accurate Greek approximations

### When to Use Binomial vs Black-Scholes

**Use Binomial Model When:**
- Pricing American options
- Handling discrete dividends
- Pricing path-dependent or exotic options
- Need intuitive understanding of option mechanics
- Working with students or explaining concepts

**Use Black-Scholes When:**
- Pricing European options on non-dividend stocks
- Need analytical Greeks
- Require very fast computation
- Working with liquid, standard options

### Advantages of Binomial Model
1. Can price American options and early exercise features
2. Easily modified for dividends, changing volatility, etc.
3. Intuitive and easy to explain
4. Provides natural framework for computing Greeks
5. Numerically stable and straightforward to implement

### Limitations
1. Slower than closed-form solutions like Black-Scholes
2. Requires choice of step count (trade-off between speed and accuracy)
3. Assumes constant volatility within each period
4. Discrete approximation introduces discretization error
5. Large trees can be memory-intensive

### Extensions and Further Study

The binomial model serves as foundation for more advanced topics:

- **Trinomial trees**: Three branches per node for improved accuracy
- **Implied trinomial trees**: Calibrating to market volatility smiles
- **Monte Carlo simulation**: Continuous-time analog using random walks
- **Finite difference methods**: PDE approaches to option pricing
- **Jump-diffusion models**: Adding discontinuous price movements

### Academic References

1. **Cox, J.C., Ross, S.A., and Rubinstein, M. (1979)**. "Option pricing: A simplified approach." *Journal of Financial Economics*, 7(3), 229-263.
   - Original paper introducing the binomial model

2. **Black, F., and Scholes, M. (1973)**. "The pricing of options and corporate liabilities." *Journal of Political Economy*, 81(3), 637-654.
   - Foundational Black-Scholes-Merton model

3. **Merton, R.C. (1973)**. "Theory of rational option pricing." *Bell Journal of Economics and Management Science*, 4(1), 141-183.
   - Theoretical foundations and early exercise conditions

4. **Hull, J.C. (2022)**. *Options, Futures, and Other Derivatives*, 11th Edition. Pearson.
   - Comprehensive textbook covering binomial trees (Chapters 13, 21)

5. **Shreve, S.E. (2004)**. *Stochastic Calculus for Finance I: The Binomial Asset Pricing Model*. Springer.
   - Rigorous mathematical treatment of binomial model

6. **Rendleman, R.J., and Bartter, B.J. (1979)**. "Two-state option pricing." *Journal of Finance*, 34(5), 1093-1110.
   - Early development of binomial approach

7. **Wilmott, P., Howison, S., and Dewynne, J. (1995)**. *The Mathematics of Financial Derivatives*. Cambridge University Press.
   - Mathematical foundations of derivative pricing

### Next Steps in Your Learning Journey

1. **Master the code**: Implement the binomial model from scratch without looking at the solutions
2. **Explore extensions**: Add dividend payments, time-varying volatility, or multiple underlying assets
3. **Study Black-Scholes**: Understand the PDE approach and its connection to risk-neutral valuation
4. **Learn Monte Carlo**: Understand simulation-based pricing for path-dependent options
5. **Apply to real markets**: Download option chain data and compare model prices to market prices
6. **Risk management**: Practice delta-hedging strategies with real-time market data

### Congratulations!

You have completed a comprehensive study of the binomial option pricing model. You now have the tools to:
- Price vanilla and American options
- Compute option Greeks
- Understand the fundamental principles of derivatives pricing
- Apply these concepts to real-world trading and risk management

Keep practicing, and happy pricing!

In [ ]:
# Solution Space for Exercise 2
print("Exercise 2 Solution:")
print("=" * 60)

# Your code here
# Hint: Use binomial_tree_option() and black_scholes_price()



In [ ]:
# Solution Space for Exercise 1
print("Exercise 1 Solution:")
print("=" * 60)

# Your code here
# Hint: Use the one_period_binomial() function

